<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP12B/MP12B_stability_prediction.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP12B · Stability Prediction after a Contingency

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Recurrent Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'problem_12_2.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12_power_grid/")
PROJECT = (['mp12b.py', 'grid39c.py', 'MP12B_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP12B/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex12_power_grid", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex12_power_grid", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'problem_12_2', 'mp12b']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import problem_12_2 as hs
import mp12b as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP12B_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

pb = hs                                                  # Ex12.2's problem_12_2.py, by its name in Ex12.2
import grid39c as g                                      # the IEEE 39-bus grid with classical machines and converters

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

Ex12.2 and MP12A ask, **before** a fault, how long a fault the grid can withstand. This mini project asks the question that comes **after** it: the fault has happened, the protection has opened the line, and the machines are swinging. **Will they stay in synchronism, and how far will they swing?**

The answer is wanted fast. If a machine is about to lose synchronism, a protection scheme can still act - trip a machine, shed a load, open a line to separate the grid in a controlled way - but only within a few hundred milliseconds. And the size of the swing decides whether the grid is close to the edge even when it holds. **Phasor measurement units** (PMUs) report the voltage phasors at the machines' buses fifty times a second, from which the rotor angles and speeds of the machines are estimated. After the first **200 ms** of them, an operator wants the next 2.8 s - long before they arrive.

The classical answer is to simulate: take the measured state at 200 ms and integrate the swing equations of the grid after the contingency. A network trained on simulated cases can answer from the same measurements in a fraction of the time. **This mini project asks how accurately and how much faster** a recurrent network predicts the rotor angles of the IEEE 39-bus grid's machines over 3 s, and the verdict, from 200 ms of noisy measurements, with 0 to 60 % of the generation from converters.

## A2 · The grid, and what is new against Ex12.2

The grid, the machines, the loads and the converters are MP12A's (and the network of Ex12.2 on a larger grid): the **IEEE 39-bus system** (MATPOWER's `case39`, taken at 50 Hz, assumed), ten **classical machines** with $H$ and $x'_d$ of Pai (1989) and **damping assumed** $D = H$, loads as **constant impedances** (assumed), and at every operating point **0, 20, 40 or 60 %** of the generation from **converters** placed at random among the nine local machines - each a negative constant impedance with no inertia (assumed; MP12A section A2 says why). A contingency is a solid fault at the first bus of one of the 35 branches whose loss leaves the grid whole, a reactance of 1e-04 p.u. to earth, cleared by tripping that branch.

**New against Ex12.2 and MP12A:**

| | value |
|---|---|
| a case | an operating point, one of the 35 contingencies, and a **clearing time between 60 and 260 ms** (uniform): some cases hold, some do not |
| the measurements | the rotor angles of machines 2 to 10 against machine 1, and the speeds of all ten, every 20 ms (50 frames per second, a PMU's rate) from the clearing to 200 ms after it: 11 frames |
| their noise | Gaussian, 0.5° on an angle and 5 mHz on a speed (assumed: a PMU-based estimate of the rotor angle; IEEE C37.118 allows 5 mHz of frequency error in steady state) |
| what is predicted | the rotor angles of machines 2 to 10 against machine 1 from 240 ms to 3 s after clearing, every 40 ms (70 instants), and the verdict: two machines more than 180° apart within 3 s |
| the data | 52 200 training cases (1 740 operating points) made in part B; 1 800 test cases (60 other operating points) in the truth file |

The cell below draws the grid with one test case's converters and contingency, and that case's measurements.

In [ ]:
# The grid with one test case, and its 200 ms of measurements
D = np.load("MP12B_truth.npz")                            # the test cases and their ground truth (part C)
n = 44                                                    # one test case
i, j = D["op_id"][n], D["cont_id"][n]
g.draw_grid(D["converters"][i], g.CONTINGENCIES[j],
            title=f"test case {n}: {100*D['share_actual'][i]:.0f} % converters, fault cleared after {1000*D['t_clear'][n]:.0f} ms")
plt.show()
tw, act = D["t_window"], ~D["converters"][i]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
for m in np.nonzero(act[1:])[0]:
    ax[0].plot(1000 * tw, D["window_angles"][n, :, m], "o", ms=3, label=f"machine {m+2}")
    ax[0].plot(1000 * tw, D["window_true_angles"][n, :, m], "k-", lw=0.5)
for m in np.nonzero(act)[0]:
    ax[1].plot(1000 * tw, 1000 * D["window_freq"][n, :, m], "o-", ms=3, lw=0.5)
ax[0].set_xlabel("time after clearing [ms]"); ax[0].set_ylabel("rotor angle against machine 1 [deg]"); ax[0].legend(fontsize=6, ncol=2)
ax[1].set_xlabel("time after clearing [ms]"); ax[1].set_ylabel("speed above 50 Hz [mHz]")
plt.tight_layout(); plt.show()
print(f"{len(D['op_id'])} test cases of {len(D['share'])} operating points; this one: contingency {j}, "
      f"fault at bus {g.CONTINGENCIES[j][0]+1}; verdict: {'loses synchronism' if D['unstable'][n] else 'stays in synchronism'}")

**What you should see.** The grid with this case's converters (squares) and its fault (star); then its eleven frames: the measured angles (dots) scattered by half a degree about the true ones (lines), and the speeds of the machines that swing, a few hundred mHz above 50 Hz just after clearing and falling. The prediction starts where these frames stop.

## A3 · The equations and their conditions

The swing equations of MP12A section A3 (Ex12.2, L12.2's *Transient Stability*), for every machine $i$ that is not a converter:

$$\frac{d\delta_i}{dt} = \omega_i, \qquad \frac{2H_i}{\omega_s}\frac{d\omega_i}{dt} = P_{m,i} - P_{e,i}(\boldsymbol{\delta}) - D_i\,\frac{\omega_i}{\omega_s}, \qquad
P_{e,i} = \sum_j E_i E_j \big(G_{ij}\cos\delta_{ij} + B_{ij}\sin\delta_{ij}\big)$$

on the **post-fault** network reduced onto the machines' internal nodes, $G_{ij} + jB_{ij}$.

| when | condition | why it holds |
|---|---|---|
| before the fault | the angles of the power flow, speeds zero, $P_{m,i} = P_{e,i}$ | the grid is in equilibrium; the turbines' power does not change within 3 s (no governor) |
| fault on, $0 < t < t_c$ | the network with the fault | it decides the state at clearing, which the measurements then see |
| after clearing | the network without the tripped branch; the state at 200 ms is what the PMUs measured | the protection has acted; from here the equations decide |
| a converter | no angle, no speed: an impedance | no rotating mass |
| the verdict | unstable if two machines are more than 180° apart within 3 s of clearing | Ex12.2's criterion |

The cell below draws this case's true angles over the 3 s, and the clearing times of all test cases against their verdict.

In [ ]:
# The whole 3 s of the case above, and the verdicts of the test cases against their clearing time
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for m in np.nonzero(act[1:])[0]:
    ax[0].plot(D["t_out"], D["traj"][n, :, m], label=f"machine {m+2}")
ax[0].axvspan(0, 0.2, color="0.85", label="measured window")
ax[0].set_xlabel("time after clearing [s]"); ax[0].set_ylabel("rotor angle against machine 1 [deg]"); ax[0].legend(fontsize=6, ncol=2)
share = D["share"][D["op_id"]]
for s in mp.SHARES:
    m = share == s
    tc = np.linspace(0.06, 0.26, 11)
    frac = [D["unstable"][m & (D["t_clear"] >= a) & (D["t_clear"] < b)].mean() for a, b in zip(tc[:-1], tc[1:])]
    ax[1].plot(1000 * (tc[:-1] + 0.01), 100 * np.array(frac), "o-", ms=3, label=f"{100*s:.0f} % converters")
ax[1].set_xlabel("clearing time [ms]"); ax[1].set_ylabel("cases that lose synchronism [%]"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()
print(f"{100*D['unstable'].mean():.1f} % of the test cases lose synchronism; by converter share: "
      + ", ".join(f"{100*s:.0f} %: {100*D['unstable'][share == s].mean():.1f} %" for s in mp.SHARES))

**What you should see.** Left: the machines swing apart after the fault, turn, and swing back, the swings decaying slowly over the 3 s; the grey band is all the network will see. Right: the longer the fault, the more cases lose synchronism - few below 100 ms, most above 220 ms - and the converters move the curve, unevenly, as in MP12A. 20.1 % of the test cases lose synchronism (16.4, 23.6, 27.8 and 12.4 % at 0, 20, 40 and 60 % converters).

---

## A4 · Why there is no exact solution

The swing equations of several coupled machines are nonlinear and have no solution in closed form (MP12A section A4); only one machine against an infinite bus, without damping, has one (the equal area criterion), and it is the solver's check in part C. The ground truth is a converged simulation.

## A5 · What is asked

| quantity | on | required |
|---|---|---|
| the verdict: will two machines separate by more than 180° within 3 s? | the 1 800 test cases, whose operating points you do not train on | at most **1 %** of the cases that lose synchronism missed, at most **5 %** of the cases that hold flagged (false alarms) |
| the rotor angles of machines 2 to 10 from 240 ms to 3 s | the test cases that hold | the largest angle error over the 3 s of each case: median within **5°**, 90 % of the cases within **15°** |
| the time to answer a new case | one Colab T4 runtime | at least **10 times shorter** than the reference model's (part C measures it) |

One network, trained once, offline; its online answer for a case it has not seen is timed against the reference model, which must simulate the 2.8 s that follow the window.

**Why these numbers, and what was relaxed.** The reference model - a simulation from the last measured frame, with its noise - misses no verdict and is within 1.15° of the truth in the median case, 2.28° in 90 % of them (part C). The network was asked first to match it. A network of the kind part D suggests, trained on the 52 200 cases, did not: its largest angle error was about 4° in the median case and 10° in the 90th percentile, mostly in the large swings near the stability limit. The agreed accuracy of the trajectories is therefore set where both can meet it, at 5° and 15°; the verdict's is set at what a protection scheme can use. A swing predicted within 15° says how close the grid came to the 180° limit; a prediction to 1° needs the simulation.

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

For one case the network takes two things and returns two:

$$\big(\text{the window},\ \text{the context}\big)\ \longmapsto\ \big(\Delta\boldsymbol\delta(t),\ \text{the verdict}\big)$$

| | what | shape |
|---|---|---|
| the window | the 11 frames, each the 9 measured angles [deg] and the 10 measured speeds [Hz], scaled | `(n, 11, 19)` |
| the context | what is known of the case besides the measurements: which machines are converters (10), the machines' pre-fault power (10), which contingency (35, one 1 among zeros), the clearing time (1), and the **grid features** of section B2 (129), scaled | `(n, 185)` |
| returns | the change of each angle since the last measured frame, at the 70 instants from 240 ms to 3 s, in units of 10°; and one number whose sign is the verdict (a logit: positive for losing synchronism) | `(n, 70, 9)`, `(n,)` |

The angle at an instant is therefore the last measured angle plus ten times the network's output: the network learns how the angles move on from where they were measured. A converter's angle is zero and is not scored.

## B2 · The physics the network is given

This network is **not** a PINN: it learns from simulated trajectories, and no equation appears in its loss. The physics is given to it in two other places:

* the **labels**: the swing equations solved by the ground truth;
* the **grid features**: what the operator's model of the grid says about the case, 129 numbers computed from the post-fault network reduced onto the machines - one Kron reduction, no simulation (`mp.grid_features`): the couplings $E_iE_jB_{ij}$ (45) and $E_iE_jG_{ij}$ (45) between every pair of machines and each machine's own $E_i^2G_{ii}$ (10), the mechanical powers $P_{m,i}$ (10), the inertias $2H_i/\omega_s$ (10), and the pre-fault angles (9). They are exactly the coefficients of the swing equations of section A3: with them and the measured state the network has, in principle, everything the simulation has.

**The loss** has two terms: the mean squared error of the predicted angles, over the cases that hold and the machines that swing (a case that loses synchronism has angles that run away, which nothing needs to predict), and the binary cross-entropy of the verdict over all cases.

## B3 · The training data

The training cases are made here, by the ground truth's own simulation (`mp.training_data`): 1 740 operating points (load 0.85 to 1.15 of the base case, each load jittered by 10 %, converters at 0 to 60 % placed at random), 30 random contingencies each with a clearing time from 60 to 260 ms, simulated by RK4 on 2 ms; the measurements get their noise from a fixed seed. The test cases in the truth file were made the same way from other seeds. About 7 to 9 minutes on a CPU.

In [ ]:
# The training data: 52 200 cases simulated here
t0 = time.perf_counter()
TR = mp.training_data()                                   # a dict of arrays, like the truth file's
print(f"made in {time.perf_counter() - t0:.0f} s")
for k in ("window_angles", "window_freq", "traj", "unstable", "grid_features"):
    print(f"{k:>15}: {TR[k].shape}")
peak = np.abs(TR["traj"][~TR["unstable"]]).max((1, 2))
print(f"in the cases that hold, the largest angle against machine 1: median {np.median(peak):.0f} deg, 90 % below {np.percentile(peak, 90):.0f} deg")

**What you should see.** 52 200 cases in about nine minutes on the authors' shared CPU, 24.1 % of them losing synchronism. In the cases that hold, the largest angle reaches 80° in the median case and more than 121° in one case in ten: the swings to predict are large, and the measurement noise of 0.5° is small against them.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Two solvers, two jobs

| | the **reference model** | the **ground truth** |
|---|---|---|
| what it is | what an engineer would run when the window closes | the same equations, from the fault's start, without noise |
| where it starts | the last measured frame, at 200 ms, with its noise | the pre-fault equilibrium; the fault and its clearing are simulated |
| time step of RK4 | 10 ms (half a cycle at 50 Hz) | 2 ms |
| its job | to be **timed** against the network: the online cost of a new case | to **score** the network and the reference model |

## C2 · The scheme, step by step

1. **The post-fault network**: the loads and converters as impedances at their pre-fault voltage, the tripped branch removed, every bus eliminated (Kron reduction), leaving a 10 × 10 matrix (`g.case_networks`).
2. **The state at the start**: for the reference model, the angles and speeds of the last frame (200 ms after clearing); for the ground truth, the pre-fault angles, then the fault simulated and cleared exactly at its clearing time.
3. **RK4 in time** (`g.simulate`), recording the angles every 40 ms to 3 s after clearing, and the largest angle between two machines that swing.
4. **The verdict**: unstable if that angle passed 180°.

The reference model is the same simulation; its error comes from where it starts - the measurement noise of the last frame - and not from its 10 ms step, as the next cells show.

In [ ]:
# The ground truth's convergence, from the truth file, and the two checks of MP12A run here
for a, b, ch in zip(D["conv_steps"][:-1], D["conv_steps"][1:], D["conv_rows"]):
    print(f"RK4 step {1000*a:3.0f} -> {1000*b:3.0f} ms: largest change of an angle on 200 test cases {ch:.2e} deg")
print("\none machine against an infinite bus (MP12A section A4):")
g.smib_check()
print("the transient energy after clearing, lossless and undamped:")
for k in (44, 528):
    i, j = D["op_id"][k], D["cont_id"][k]
    g.energy_check(mp.op_from_arrays(D, i), D["converters"][i], *g.CONTINGENCIES[j], t_clear=float(D["t_clear"][k]))

**What you should see.** Halving the step from 10 ms changes no angle by more than 6e-05°, and below 2 ms the changes are at the resolution of the stored single-precision angles (about 1e-05° at 100°): the ground truth's 2 ms is converged far beyond anything scored. The one-machine case returns the equal area criterion's clearing time within 0.02 ms, and the transient energy of the lossless, undamped grid is conserved to 4e-11 of the kinetic energy at clearing or better.

## C3 · An independent check: ANDES

As in MP12A, ANDES (Cui, Li and Tomsovic, *IEEE Trans. Power Systems*, 2021) was given eight stable test cases - the same network, operating point, classical machines (GENCLS), loads and converters, the fault as a reactance of 1e-04 p.u. - and integrated them with its own implicit trapezoidal method on 2 ms. The angles it returned are in the truth file.

In [ ]:
# ANDES's angles against the ground truth's, from the truth file
for k, a, xf in zip(D["andes_cases"], D["andes_angles"], D["andes_xf"]):
    e = np.abs(a - D["traj"][k]).max()
    print(f"test case {k:4d} ({100*D['share'][D['op_id'][k]]:.0f} % converters, cleared after {1000*D['t_clear'][k]:.0f} ms): "
          f"largest angle difference over 3 s {e:.2f} deg" + ("" if xf == 1e-4 else f"   (ANDES needed a milder fault, {xf:.0e} p.u.)"))

**What you should see.** In seven of the eight cases ANDES's angles agree with the ground truth's to 0.01° over the whole 3 s: the same model in two simulators, integrated by two different methods. In one, ANDES could not converge after clearing with the 1e-04 p.u. fault and was run with a milder 1e-03 p.u. one; it then differs by 2.88°, the effect of the milder fault.

## C4 · The reference model: accuracy and time

The truth file holds the reference model's angles and verdict for every test case, started from the noisy last frame, and for comparison from the true state. The cell below scores both, then runs the reference model on three test cases on this runtime: the same numbers, and the online time of a new case.

In [ ]:
# The reference model against the ground truth on the 1800 test cases, and its time for a new case
K = int(D["k_pred"])                                      # the prediction starts at t_out[6] = 0.24 s
act_all = ~D["converters"][D["op_id"]][:, 1:]             # machines 2 to 10 that swing, per case
hold = ~D["unstable"]
def largest_error(pred):
    # the largest angle error of each case over 0.24 - 3 s and the machines that swing [deg]
    return (np.abs(pred - D["traj"][:, K:]) * act_all[:, None, :]).max((1, 2))
for key, name in (("ref_traj", "from the noisy last frame"), ("ref_traj_exact_start", "from the true state")):
    e = largest_error(D[key])[hold]
    print(f"reference model {name}: largest angle error, median {np.median(e):.2f} deg, 90 % within {np.percentile(e, 90):.2f} deg, "
          f"worst {e.max():.1f} deg")
r = D["ref_unstable"]; u = D["unstable"]
print(f"verdicts from the noisy frame: missed {(u & ~r).sum()} of {u.sum()} cases that lose synchronism, "
      f"false alarms {(~u & r).sum()} of {(~u).sum()}")
times = []
for k in (44, 528, 1215):
    i, j = D["op_id"][k], D["cont_id"][k]
    op, conv = mp.op_from_arrays(D, i), D["converters"][i]
    a, _ = mp.reference(op, conv, g.CONTINGENCIES[j], D["window_angles"][k, -1], D["window_freq"][k, -1])
    assert np.abs(a - D["ref_traj"][k]).max() < 1e-3
    times.append(middle(lambda: mp.reference(op, conv, g.CONTINGENCIES[j], D["window_angles"][k, -1], D["window_freq"][k, -1])))
t_ref = float(np.median(times))
e_ref = largest_error(D["ref_traj"])[hold]
print(f"\none new case: {1000*t_ref:.1f} ms; the network must answer in under {100*t_ref:.2f} ms (ten times shorter)")
REF = {"t_ref_s": t_ref, "median_ref": float(np.median(e_ref)), "p90_ref": float(np.percentile(e_ref, 90)),
       "missed_ref": int((u & ~r).sum()), "alarms_ref": int((~u & r).sum())}          # kept for part E

**What you should see.** Started from the true state, the reference model reproduces the ground truth to the stored precision: its 10 ms step costs nothing. Started from the noisy last frame, as it must be, its largest angle error is 1.15° in the median case and within 2.28° in 90 % of them; a few cases near the stability limit, where half a degree decides between a large swing and none, are off by much more. It gets every verdict of the 1 800 right. One new case - a Kron reduction and 280 steps of RK4 - takes about 0.14 s on the authors' CPU, shared with other jobs: the bar for the network is a tenth of it.

---

# Part D · The Recurrent Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** with the verdict's **threshold** (TODO 2), and the **figures** you need to understand it (TODO 3).

**The contract.** Your network is a PyTorch module called `net`, used as `net(W, C)`:

| argument | meaning | shape |
|---|---|---|
| `W` | the window: 11 frames of 9 angles and 10 speeds, scaled | `(n, 11, 19)` |
| `C` | the context of section B1, scaled | `(n, 185)` |
| returns | the change of the angles since the last frame at the 70 instants, in units of 10°; and the verdict's logit | `(n, 70, 9)`, `(n,)` |

`THRESHOLD` is your verdict's threshold: a case is flagged as losing synchronism if its logit is above it.

## D2 · The data and the loss, given

The cell below builds the scaled windows and contexts of the training and the test cases, the targets, and the masks of the machines that swing. `batch(idx)` returns the training tensors of the cases `idx`. `loss_fn(net, idx)` returns the two terms of section B2. `predict(W, C)` gives the angles [deg] and the logits; `quick_score()` the test score: the largest angle error (median and 90th percentile) and the verdicts at your `THRESHOLD`.

In [ ]:
# The data and the loss, given -------------------------------------------------------
def context(S):
    """The 185 numbers of section B1 for every case of S (the training dict or the truth file)."""
    conv = S["converters"][S["op_id"]].astype(float)
    return np.concatenate([conv, S["op_Pg"][S["op_id"]] / 10, np.eye(g.N_CONT)[S["cont_id"]],
                           (S["t_clear"] / 0.1)[:, None], S["grid_features"].astype(float)], 1)

def window(S):
    return np.concatenate([S["window_angles"] / 100.0, S["window_freq"] / 0.2], -1).astype(float)   # (n, 11, 19)

Wtr, Ctr, Wte, Cte = window(TR), context(TR), window(D), context(D)
mu_w, sd_w = Wtr.reshape(-1, 19).mean(0), Wtr.reshape(-1, 19).std(0) + 1e-9
mu_c, sd_c = Ctr.mean(0), Ctr.std(0) + 1e-9
W_tr, C_tr = torch.tensor((Wtr - mu_w) / sd_w), torch.tensor((Ctr - mu_c) / sd_c)
W_te, C_te = torch.tensor((Wte - mu_w) / sd_w, device=DEVICE), torch.tensor((Cte - mu_c) / sd_c, device=DEVICE)
last_tr = TR["window_angles"][:, -1, :].astype(float)     # the last measured angles [deg]
Y_tr = torch.tensor((TR["traj"][:, K:].astype(float) - last_tr[:, None, :]) / 10.0)   # the targets, in 10 deg
U_tr = torch.tensor(TR["unstable"].astype(float))
M_tr = torch.tensor((~TR["converters"][TR["op_id"]][:, 1:]).astype(float))   # machines 2 to 10 that swing
train_idx = np.arange(len(U_tr))

def batch(idx):
    return W_tr[idx].to(DEVICE), C_tr[idx].to(DEVICE)   # onto the GPU when there is one

def loss_fn(net, idx):
    dth, logit = net(*batch(idx))
    w = ((1 - U_tr[idx])[:, None, None] * M_tr[idx][:, None, :]).to(DEVICE)   # cases that hold, machines that swing
    traj = ((dth - Y_tr[idx].to(DEVICE)) ** 2 * w).sum() / (w.sum() * dth.shape[1] + 1e-9)   # mean squared error, targets on the device
    verdict = torch.nn.functional.binary_cross_entropy_with_logits(logit, U_tr[idx].to(DEVICE))   # the verdict's loss
    return traj, verdict

def predict(W, C, last, chunk=4000):
    """The angles [deg] and the verdict's logits of the cases (W, C), in chunks of 4000 cases."""
    net.eval()                                            # evaluation mode
    dth, logit = [], []
    with torch.no_grad():                                 # evaluation only: no gradients
        for s in range(0, len(W), chunk):
            a, b = net(W[s:s + chunk], C[s:s + chunk])
            dth.append(a.cpu().numpy()); logit.append(b.cpu().numpy())
    net.train()                                           # back to training mode
    return last[:, None, :] + 10 * np.concatenate(dth), np.concatenate(logit)

THRESHOLD = 0.0                                           # your verdict's threshold; set it in TODO 2
def quick_score():
    pred, logit = predict(W_te, C_te, D["window_angles"][:, -1, :].astype(float))
    e = largest_error(pred)[hold]
    flag = logit > THRESHOLD
    return (f"largest angle error median {np.median(e):.2f} deg, 90 % {np.percentile(e, 90):.2f} deg; "
            f"missed {(u & ~flag).sum()} of {u.sum()}, false alarms {100*(~u & flag).sum()/(~u).sum():.1f} %")

## D3 · TODO 1 · Your network

Write the class `Net` and make `net = Net().to(DEVICE)`. Decide:

* **how the window is read.** A recurrent layer (a GRU or an LSTM, L5.2) reads the 11 frames in order and can average the noise away; a dense network on the 209 flattened numbers is the simpler alternative.
* **how the context enters.** As the recurrent layer's initial state, beside its last state in the decoder, or both.
* **the decoder.** From the window's summary and the context to 630 angle changes and one logit: how wide, how deep.

What the data tell you: angles that swing by about 80° in the median case, measured to 0.5°; one case in five loses synchronism; the grid features carry the coefficients of the swing equations.

In [ ]:
# TODO 1 - your network. net(W, C) -> (angle changes / 10 deg (n, 70, 9), verdict logit (n,))
import torch.nn as nn

class Net(nn.Module):                                     # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        ...                                              # your layers

    def forward(self, W, C):
        ...                                              # return the angle changes and the logit

torch.manual_seed(88)                                     # fix the random start so a run repeats
net = Net().to(DEVICE)                                    # the network, on the GPU when there is one
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training, and your threshold

Train `net` with `loss_fn` on the training cases. Decide the optimiser, the steps, the learning rate and its schedule, the batch size and the weight of the verdict's term. Keep the loss history for TODO 3 and **measure the training time** in `TRAIN_SECONDS`.

Then choose `THRESHOLD` **on the training cases only**: low enough that the cases losing synchronism are flagged, high enough that the false alarms stay within 5 %. The test cases score it.

In [ ]:
# TODO 2 - your training, and your threshold
history = []
t0 = time.perf_counter()
...                                                      # your optimiser and loop; append the losses to history
TRAIN_SECONDS = time.perf_counter() - t0
THRESHOLD = ...                                          # chosen on the training cases
print(f"trained in {TRAIN_SECONDS:.0f} s; threshold {THRESHOLD}; test: {quick_score()}")

## D5 · TODO 3 · Your figures

Draw what you need to understand your network and to answer the report: at least the loss terms during training, and the predicted against the true angles of a few test cases that hold, one of them with a large swing. Save each figure you want in the report as `MP12B_report_<name>.png` (in `cc.output_path`): part E embeds them in the PDF.

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores, and keeps the answers in the dictionary `R`:

* the angles and the verdict of the 1 800 test cases, and their largest angle errors;
* the **online time** of a new case on this runtime: everything the network needs after the window closes - the grid features (one Kron reduction), the scaling, and one call of `net` - against the reference model on the same case, on three test cases; and the network for all 1 800 at once.

In [ ]:
# The evaluation, given ------------------------------------------------------------
last_te = D["window_angles"][:, -1, :].astype(float)
pred, logit = predict(W_te, C_te, last_te)
R = {"pred": pred, "logit": logit, "flag": logit > THRESHOLD, "threshold": THRESHOLD, "error": largest_error(pred)}

def answer(k):
    """The network's answer to test case k, from its window and what the operator knows: what it computes online."""
    i, j = D["op_id"][k], D["cont_id"][k]
    gf = mp.grid_features(mp.op_from_arrays(D, i), D["converters"][i], g.CONTINGENCIES[j]).astype(np.float32)   # stored in single precision
    c = np.r_[D["converters"][i].astype(float), D["op_Pg"][i] / 10, np.eye(g.N_CONT)[j], D["t_clear"][k] / 0.1, gf]
    w = np.concatenate([D["window_angles"][k] / 100.0, D["window_freq"][k] / 0.2], -1)
    with torch.no_grad():                                 # evaluation only: no gradients
        dth, lg = net(torch.tensor((w - mu_w) / sd_w, device=DEVICE)[None], torch.tensor((c - mu_c) / sd_c, device=DEVICE)[None])
    return D["window_angles"][k, -1] + 10 * dth[0].cpu().numpy(), float(lg[0])

t_net, t_ref = [], []
for k in (44, 528, 1215):
    a, lg = answer(k)
    assert np.abs(a - pred[k]).max() < 1e-4
    i, j = D["op_id"][k], D["cont_id"][k]
    op, conv = mp.op_from_arrays(D, i), D["converters"][i]
    t_net.append(middle(lambda: answer(k)))
    t_ref.append(middle(lambda: mp.reference(op, conv, g.CONTINGENCIES[j], D["window_angles"][k, -1], D["window_freq"][k, -1])))
R["net_1"], R["ref_1"] = float(np.median(t_net)), float(np.median(t_ref))
R["net_all"] = middle(lambda: predict(W_te, C_te, last_te))
R["train_seconds"] = TRAIN_SECONDS
R["n_weights"] = sum(p.numel() for p in net.parameters())   # number of trainable weights and biases
R["device"] = DEVICE
torch.save(net.state_dict(), cc.output_path("MP12B_net.pt"))   # the trained weights, beside the report
print(f"online, one new case: network {1000*R['net_1']:.2f} ms, reference {1000*R['ref_1']:.1f} ms ({R['ref_1']/R['net_1']:.0f} times)")
print(f"the network on all 1800 test cases at once: {1000*R['net_all']:.0f} ms; training {TRAIN_SECONDS:.0f} s offline")

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the ground truth `D` and the reference model's results `REF` from part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the ten questions.

In [ ]:
# What part E uses, all from the parts above
print("in R:", ", ".join(R))
print(f"reference model (part C): largest angle error median {REF['median_ref']:.2f} deg, 90 % {REF['p90_ref']:.2f} deg; "
      f"{1000*REF['t_ref_s']:.1f} ms per new case")

## E2 · TODO 4 · The goal metrics

Compute, against the ground truth, and put in the dictionary `METRICS` (a name and a number each; the report prints them as its results table):

* the cases losing synchronism that your verdict missed, and the false alarms in per cent of the cases that hold;
* the largest angle error of the test cases that hold: median, 90th percentile, worst [deg];
* the same median at each converter share;
* the network's online time for a new case, the reference model's on the same runtime, and their ratio;
* the offline training time.

Then say in one line each whether the three requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>55}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP12B_report_<name>.png` in `cc.output_path`, the figures your answers refer to. At least: the distribution of the largest angle error of the network and of the reference model; the logits of the test cases, those that hold against those that do not, with your threshold; and the largest error against the size of the swing.

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the ten questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 7 are about your network and its result, 8 and 9 ask you to use it, 10 concludes. The next cell writes `MP12B_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: how it reads the window, how the context enters, the decoder, and the number of weights. Why this design for 11 noisy frames and 630 angles to predict?": """
""",
    "2. What did the grid features of section B2 change? Give the median and 90th-percentile largest angle errors and the missed verdicts of your network with and without them (one training each), and say why.": """
""",
    "3. Your training: optimiser, steps, batch size, learning rate and schedule, the weight of the verdict's term, and the offline time on the T4. How did you choose the threshold on the training cases, and what is it?": """
""",
    "4. The verdict on the 1 800 test cases: missed cases and false alarms in per cent, against the reference model's. Are they within 1 % and 5 %? What do the missed cases have in common (clearing time, contingency, converter share)?": """
""",
    "5. The angles of the test cases that hold: your median, 90th-percentile and worst largest angle error, against the reference model's 1.15 and 2.28 deg. Where in the 3 s and in which cases is your error largest, and why there?": """
""",
    "6. The converter shares: your median largest error and missed verdicts at 0, 20, 40 and 60 % converters. Which share is hardest for the network, and why?": """
""",
    "7. The online time of a new case: your network against the reference model on the same runtime, in ms, and the ratio; and the network for all 1 800 test cases at once. What does each spend its time on?": """
""",
    "8. Shorten the window: feed your network (or one retrained) only the first 6 frames, 100 ms, padded with the last one. How do the verdict and the median angle error change? How early could a protection scheme act on your prediction?": """
""",
    "9. Train again with another seed. How much do the missed verdicts, the false alarms and the median and 90th-percentile errors of the two runs differ? Is your network reproducible within the requirements?": """
""",
    "10. To conclude: would you let this network trigger a protection scheme, or only warn an operator? Say what it may decide alone, when the simulation must still be run, and what you would check before trusting it on a grid with more converters.": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" if abs(v) >= 0.01 else f"| {k} | {v:.2e} |" for k, v in METRICS.items()]
    out = ["# MP12B — Stability Prediction after a Contingency", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP12B_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP12B_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP12B_report.md into MP12B_report.pdf, with any figure
# saved as MP12B_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP12B_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP12B_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP12B_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP12B_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass